In [ ]:
# Cell 1: Install Required Packages

!pip install ASE
!pip install mace-torch ase rdkit weas-widget

In [ ]:
# Cell 2: Import Required Libraries

import numpy as np                              # Computational library
import pandas as pd                             # Excel of python

from rdkit import Chem                          # Used to build molecules (basic package)
from rdkit.Chem import AllChem                  # Used to build molecules (advanced package)

from mace.calculators import mace_off           # MACE-OFF (Machine Learning Potential)

# Atomic Simulation Environment Libraries
from ase import Atoms                           # Represents a molecule object with information
from ase.build import molecule                  # Creates an atomic structure from the database
from ase.optimize import QuasiNewton            # Optimization / energy minimization
from ase.vibrations import Vibrations           # Used to calculate vibrational modes of the Atom object
from ase.thermochemistry import IdealGasThermo  # Allows you to calculate entropy, enthalpy, and gibbs free energy
from ase.units import kJ, mol                   # Conversion for units

In [ ]:
# Cell 3: Load MACE-OFF

print("Loading MACE-OFF (medium model)...")
calc_mol = mace_off(model="small", default_dtype="float64")
print("MACE-OFF loaded.")

In [ ]:
# Cell 4: Calculate Propanol (CCCO) Chemical Properties
# Build the Molecule using SMILES

smiles = 'CCCO' # SMILES for Propanol (CCCO)
seed = 42

mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol) # Adds explicit hydrogens to the molecule
AllChem.EmbedMolecule(mol, randomSeed=seed) # Randomly places atoms at correct distances from each other
AllChem.MMFFOptimizeMolecule(mol) # Optimizes the geometry using the MMFF94 classical force field (Starting Position for MACE-OFF optimization)
conf = mol.GetConformer()
symbols = [a.GetSymbol() for a in mol.GetAtoms()]
positions = conf.GetPositions()

atoms_CCCO = Atoms(symbols=symbols, positions=positions)

# Same as last code except atoms_CCCO is defined instead of being called from g2 list
atoms_CCCO.calc = calc_mol # Calls to MACE-OFF to be used
dyn = QuasiNewton(atoms_CCCO, logfile=None)
dyn.run(fmax=0.01) # Finding energy minimum
potentialenergy_CCCO = atoms_CCCO.get_potential_energy() # Get potential energy

vib = Vibrations(atoms_CCCO, name='ccco_vib')
vib.clean()
vib.run()
vib_energies = vib.get_energies() # Gets vibrational energy based on molecule geometry
vib_energies = np.array([e.real for e in vib_energies if e.real > 0.01]) # Filters out imaginary numbers and very low frequencies

# Takes inputs of vibrational energies, potential energy, and geometry to compute chemical properties
thermo = IdealGasThermo(
    vib_energies=vib_energies,
    potentialenergy=potentialenergy_CCCO,
    atoms=atoms_CCCO,
    geometry='nonlinear', # Linear (Straight Line) or Nonlinear (Bent in any way)
    symmetrynumber=1, # How many times you can rotate the molecule and get the same configuration
    spin=0, # 0.5 for each unpaired electrons
)


# Records Chemical Property Data at 6 Temperatures at 1 atmosphere and Displays it
temps = [298.15, 400, 500, 600, 700, 800]
P = 101325.

records = []
for T in temps:
    H = thermo.get_enthalpy(T, verbose=False)
    S = thermo.get_entropy(T, P, verbose=False)
    G = thermo.get_gibbs_energy(T, P, verbose=False)
    records.append({"T (K)": T, "H (eV)": H, "S (eV/K)": S, "G (eV)": G})

df1 = pd.DataFrame(records)
display(df1)